# <b> Problemen bij stemherkenning en diarisatie oplossen: overlap </b>

## Testaudio6 — unseen validation fragment

Doel:
controleren hoe de huidige diarization + docent recognition-pipeline werkt op nieuw audiofragment met dezelfde docentstem.

Bron:
Peter Hoefnagels-fragment, nog niet gebruikt bij eerdere test.

Belangrijk:
dit fragment wordt gebruikt als unseen test en niet om de threshold op te tunen.

In [1]:
import json
import pandas as pd
from pathlib import Path

path = Path("../../Data-local/processed/docent_recognition/testaudio6_fragment_diarized_turns_docent_roles_BEFORE.json")

with open(path, encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame([
    {
        "turn_id": t["turn_id"],
        "start": t["start"],
        "end": t["end"],
        "speaker": t["speaker"],
        "text": t["text"],
        "overlap": t["overlap"],
        "uncertain": t["uncertain_assignment"],
        "docent_role": t["docent_role"],
        "similarity": t["docent_role_similarity"],
    }
    for t in data["turns"]
])

df

,turn_id,start,end,speaker,text,overlap,uncertain,docent_role,similarity
0,0,0.00,2.60,MAIN_SPEAKER,Het gaat erom dat we mannen niet nodig hebben.,False,False,OTHER,0.2062
1,1,2.60,4.60,OTHER_SPEAKER_1,Wij zijn overbodig geworden in Maatschappij.,True,False,ONZEKER,NaN
2,2,4.60,5.60,OTHER_SPEAKER_1,"Ja, helemaal.",True,False,ONZEKER,NaN
3,3,5.60,6.60,OTHER_SPEAKER_1,Is dat serieus?,True,True,ONZEKER,NaN
4,4,6.60,13.16,MAIN_SPEAKER,Serieus. Want vroeger was het misschien nog zo...,False,False,OTHER,0.2246
5,5,13.16,15.16,OTHER_SPEAKER_1,Maar nu heb je een navigatiesysteem.,True,False,ONZEKER,NaN
6,6,15.16,16.16,MAIN_SPEAKER,Precies.,True,True,ONZEKER,NaN
7,7,16.16,17.16,MAIN_SPEAKER,Precies.,False,False,OTHER,0.1730
8,8,17.16,18.16,OTHER_SPEAKER_1,"Nee, maar u begrijpt het toch?",True,True,ONZEKER,NaN
9,9,18.16,19.16,OTHER_SPEAKER_1,We hebben onszelf eigenlijk een beetje...,False,False,DOCENT,0.3547


In [3]:
df["docent_role"].value_counts()

docent_role
ONZEKER    28
OTHER       9
DOCENT      2
Name: count, dtype: int64

### Observatie

Bij testaudio6 worden opvallend veel turns als ONZEKER gemarkeerd.
Dit komt vooral doordat `overlap=true` in de huidige docent recognition-logica
direct leidt tot ONZEKER, zonder dat speaker recognition wordt uitgevoerd.

Handmatige controle van het audiofragment laat zien dat niet alle overlap-turns
daadwerkelijk onbruikbare gemengde spraak bevatten.

Daarom wordt getest of `overlap` beter als waarschuwing behandeld in plaats van een strenge regel. Dit wordt aangepast in de code. 
Overlap is dus voor nu nog te streng gedefinieerd. 

### Opnieuw testen na aanpassing

Nieuwe regel:
- `uncertain_assignment=true` → ONZEKER
- overlap alleen → speaker recognition mag doorgaan
- overlap wordt wel als waarschuwing bewaard

In [4]:
path = Path("../../Data-local/processed/docent_recognition/testaudio6_fragment_diarized_turns_docent_roles.json")

with open(path, encoding="utf-8") as f:
    data = json.load(f)

df2 = pd.DataFrame([
    {
        "turn_id": t["turn_id"],
        "start": t["start"],
        "end": t["end"],
        "speaker": t["speaker"],
        "text": t["text"],
        "overlap": t["overlap"],
        "uncertain": t["uncertain_assignment"],
        "docent_role": t["docent_role"],
        "similarity": t["docent_role_similarity"],
    }
    for t in data["turns"]
])

df2

,turn_id,start,end,speaker,text,overlap,uncertain,docent_role,similarity
0,0,0.00,2.60,MAIN_SPEAKER,Het gaat erom dat we mannen niet nodig hebben.,False,False,OTHER,0.2062
1,1,2.60,4.60,OTHER_SPEAKER_1,Wij zijn overbodig geworden in Maatschappij.,True,False,DOCENT,0.4216
2,2,4.60,5.60,OTHER_SPEAKER_1,"Ja, helemaal.",True,False,OTHER,0.0925
3,3,5.60,6.60,OTHER_SPEAKER_1,Is dat serieus?,True,True,ONZEKER,NaN
4,4,6.60,13.16,MAIN_SPEAKER,Serieus. Want vroeger was het misschien nog zo...,False,False,OTHER,0.2246
5,5,13.16,15.16,OTHER_SPEAKER_1,Maar nu heb je een navigatiesysteem.,True,False,DOCENT,0.3639
6,6,15.16,16.16,MAIN_SPEAKER,Precies.,True,True,ONZEKER,NaN
7,7,16.16,17.16,MAIN_SPEAKER,Precies.,False,False,OTHER,0.1730
8,8,17.16,18.16,OTHER_SPEAKER_1,"Nee, maar u begrijpt het toch?",True,True,ONZEKER,NaN
9,9,18.16,19.16,OTHER_SPEAKER_1,We hebben onszelf eigenlijk een beetje...,False,False,DOCENT,0.3547


#### Voor aanpassing overlap regel

In [6]:
df["docent_role"].value_counts()

docent_role
ONZEKER    28
OTHER       9
DOCENT      2
Name: count, dtype: int64

#### Na aanpassing overlap regel

In [5]:
df2["docent_role"].value_counts()

docent_role
OTHER      23
ONZEKER    11
DOCENT      5
Name: count, dtype: int64

De oorspronkelijke overlap-regel bleek te conservatief. Na aanpassing daalde het aantal ONZEKER-labels van 28 naar 11, terwijl onzeker toegewezen turns geblokkeerd bleven.

## Testaudio7 — sanity validation fragment

Als laatste check wordt er een fragment gebruikt van Peter Hoefnagels waar er veel sprake is van overlap. 

In [9]:
path = Path("../../Data-local/processed/docent_recognition/testaudio7_fragment_diarized_turns_docent_roles.json")

with open(path, encoding="utf-8") as f:
    data = json.load(f)

df3 = pd.DataFrame([
    {
        "turn_id": t["turn_id"],
        "start": t["start"],
        "end": t["end"],
        "speaker": t["speaker"],
        "text": t["text"],
        "overlap": t["overlap"],
        "uncertain": t["uncertain_assignment"],
        "docent_role": t["docent_role"],
        "similarity": t["docent_role_similarity"],
    }
    for t in data["turns"]
])

df3

,turn_id,start,end,speaker,text,overlap,uncertain,docent_role,similarity
0,0,0.0,3.0,MAIN_SPEAKER,"Ja, dames en heren, mag ik de aantal, wat heb ...",False,False,DOCENT,0.4449
1,1,3.0,4.0,MAIN_SPEAKER,Heb je de cijfers?,False,True,ONZEKER,NaN
2,2,4.0,6.0,OTHER_SPEAKER_1,"Ja, meneer, dat doet u wel heel lang.",True,False,OTHER,0.0445
3,3,6.0,7.0,OTHER_SPEAKER_1,"Nee, helemaal niets.",True,True,ONZEKER,NaN
4,4,7.0,9.0,OTHER_SPEAKER_1,U heeft twee weken erover gedaan.,True,False,OTHER,0.3428
5,5,9.0,12.0,MAIN_SPEAKER,"Guus, echt serieus, echt niet waar wat je zegt.",True,True,ONZEKER,NaN
6,6,12.0,20.0,MAIN_SPEAKER,Waarom zeg je dingen die niet waar zijn? Vorig...,False,False,DOCENT,0.5260
7,7,20.0,22.0,OTHER_SPEAKER_1,"Nou, die wiskundige leraar doet altijd binnen ...",True,False,OTHER,-0.0018
8,8,22.0,23.0,OTHER_SPEAKER_1,Ja.,False,False,OTHER,0.0547
9,9,23.0,24.0,OTHER_SPEAKER_1,"Nee, dat is onzin.",True,True,ONZEKER,NaN


In [10]:
df3["docent_role"].value_counts()

docent_role
OTHER      12
ONZEKER     9
DOCENT      5
Name: count, dtype: int64

De onzekerheidsgraad zit hier wel iets hoger dan bij de vorige fragmenten. Dit fragment specifiek heeft ook wel een erg rumoerige klas en veel leerlingen die door de docent praten. Hier moet rekening mee gehouden worden voor het model. 